<a href="https://www.kaggle.com/code/avtnshm/residual-ssl-linear-probe?scriptVersionId=311416582" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import os
for root, _, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/notebooks/rajeshkm57/residual-ssl-stl-10-v1-4/__results__.html
/kaggle/input/notebooks/rajeshkm57/residual-ssl-stl-10-v1-4/__notebook__.ipynb
/kaggle/input/notebooks/rajeshkm57/residual-ssl-stl-10-v1-4/__output__.json
/kaggle/input/notebooks/rajeshkm57/residual-ssl-stl-10-v1-4/custom.css
/kaggle/input/notebooks/rajeshkm57/residual-ssl-stl-10-v1-4/checkpoints/D.pt
/kaggle/input/notebooks/rajeshkm57/residual-ssl-stl-10-v1-4/data/stl10_binary.tar.gz
/kaggle/input/notebooks/rajeshkm57/residual-ssl-stl-10-v1-4/data/stl10_binary/test_X.bin
/kaggle/input/notebooks/rajeshkm57/residual-ssl-stl-10-v1-4/data/stl10_binary/class_names.txt
/kaggle/input/notebooks/rajeshkm57/residual-ssl-stl-10-v1-4/data/stl10_binary/train_X.bin
/kaggle/input/notebooks/rajeshkm57/residual-ssl-stl-10-v1-4/data/stl10_binary/unlabeled_X.bin
/kaggle/input/notebooks/rajeshkm57/residual-ssl-stl-10-v1-4/data/stl10_binary/test_y.bin
/kaggle/input/notebooks/rajeshkm57/residual-ssl-stl-10-v1-4/data/stl10_binary/f

In [2]:
# ============================================================
# CONFIG
# ============================================================
EPOCHS = 500
SEED = 1

CKPTS = {
    "A": "/kaggle/input/notebooks/avtnshm/residual-ssl-stl-10-v1-1/checkpoints/A.pt",
    "B": "/kaggle/input/notebooks/avtnsh/residual-ssl-stl-10-v1-2/checkpoints/B.pt",
    "C": "/kaggle/input/notebooks/rajeshr1005/residual-ssl-stl-10-v1-3/checkpoints/C.pt",
    "D": "/kaggle/input/notebooks/rajeshkm57/residual-ssl-stl-10-v1-4/checkpoints/D.pt",
}

DATA_DIR = "/kaggle/working/data"
SAVE_PATH = "/kaggle/working/probe_results.txt"

In [4]:
import os, time, datetime
import random, numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as T
from torchvision.models import resnet18
from torchvision.datasets import STL10

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Device:", device)

Device: cuda


In [5]:
train_data = STL10(DATA_DIR, split="train", download=True)
test_data  = STL10(DATA_DIR, split="test", download=True)

transform = T.Compose([
    T.Resize(96),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],
                [0.229,0.224,0.225])
])

class ProbeDataset(Dataset):
    def __init__(self,data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self,idx):
        img, y = self.data[idx]
        return transform(img), y

train_loader = DataLoader(ProbeDataset(train_data),
                          batch_size=256, shuffle=True)

test_loader = DataLoader(ProbeDataset(test_data),
                         batch_size=256, shuffle=False)

100%|██████████| 2.64G/2.64G [03:51<00:00, 11.4MB/s] 


In [6]:
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(
            *list(resnet18(weights=None).children())[:-1]
        )

    def forward(self,x):
        return self.backbone(x).flatten(1)

In [7]:
results = {}

for MODE, CKPT_PATH in CKPTS.items():

    print(f"\n===== RUNNING {MODE} =====")

    encoder = Encoder().to(device)
    ckpt = torch.load(CKPT_PATH, map_location=device)

    encoder.load_state_dict(ckpt["model"], strict=False)

    for p in encoder.parameters():
        p.requires_grad = False

    encoder.eval()

    classifier = nn.Linear(512, 10).to(device)
    optimizer = torch.optim.Adam(classifier.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    start_time = time.time()

    # -------- TRAIN --------
    for epoch in range(1, EPOCHS+1):

        classifier.train()
        total_loss = 0
        correct = 0
        total = 0
        ep_start = time.time()

        for x,y in train_loader:
            x,y = x.to(device), y.to(device)

            with torch.no_grad():
                feat = encoder(x)

            logits = classifier(feat)
            loss = criterion(logits, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            pred = logits.argmax(1)
            correct += (pred==y).sum().item()
            total += y.size(0)

        acc = correct / total

        ep_time = time.time() - ep_start

        print(f"[{MODE}] Epoch {epoch}/{EPOCHS} | "
              f"loss {total_loss/len(train_loader):.4f} | "
              f"acc {acc:.4f} | {ep_time:.1f}s")

    # -------- TEST --------
    classifier.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x,y in test_loader:
            x,y = x.to(device), y.to(device)
            feat = encoder(x)
            logits = classifier(feat)

            pred = logits.argmax(1)
            correct += (pred==y).sum().item()
            total += y.size(0)

    test_acc = correct / total

    total_time = time.time() - start_time

    h = int(total_time // 3600)
    m = int((total_time % 3600) // 60)
    s = int(total_time % 60)

    print(f"\n[{MODE}] FINAL ACC: {test_acc:.4f} | Time: {h}h {m}m {s}s")

    results[MODE] = test_acc


===== RUNNING A =====
[A] Epoch 1/500 | loss 2.3437 | acc 0.1136 | 5.3s
[A] Epoch 2/500 | loss 2.2771 | acc 0.1402 | 4.1s
[A] Epoch 3/500 | loss 2.2402 | acc 0.1812 | 4.1s
[A] Epoch 4/500 | loss 2.2214 | acc 0.1884 | 4.0s
[A] Epoch 5/500 | loss 2.1964 | acc 0.2044 | 4.0s
[A] Epoch 6/500 | loss 2.1766 | acc 0.2132 | 4.1s
[A] Epoch 7/500 | loss 2.1599 | acc 0.2320 | 4.1s
[A] Epoch 8/500 | loss 2.1490 | acc 0.2258 | 4.1s
[A] Epoch 9/500 | loss 2.1295 | acc 0.2324 | 4.1s
[A] Epoch 10/500 | loss 2.1172 | acc 0.2420 | 4.1s
[A] Epoch 11/500 | loss 2.1059 | acc 0.2594 | 4.1s
[A] Epoch 12/500 | loss 2.0883 | acc 0.2552 | 4.0s
[A] Epoch 13/500 | loss 2.0828 | acc 0.2592 | 4.0s
[A] Epoch 14/500 | loss 2.0684 | acc 0.2852 | 4.1s
[A] Epoch 15/500 | loss 2.0538 | acc 0.2838 | 4.1s
[A] Epoch 16/500 | loss 2.0501 | acc 0.2888 | 4.1s
[A] Epoch 17/500 | loss 2.0382 | acc 0.2862 | 4.0s
[A] Epoch 18/500 | loss 2.0317 | acc 0.2798 | 4.0s
[A] Epoch 19/500 | loss 2.0317 | acc 0.2770 | 4.1s
[A] Epoch 20/500 

In [8]:
print("\n===== FINAL RESULTS =====")

for k,v in results.items():
    print(f"{k}: {v:.4f}")

with open(SAVE_PATH, "w") as f:
    for k,v in results.items():
        f.write(f"{k}: {v:.4f}\n")

print("\nSaved to:", SAVE_PATH)


===== FINAL RESULTS =====
A: 0.3950
B: 0.4138
C: 0.3980
D: 0.4085

Saved to: /kaggle/working/probe_results.txt
